In [ ]:
!pip install openai-agents pydantic --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.0/843.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 12.3 MB/s eta 0:00:00


In [ ]:
from dataclasses import dataclass
from typing import Literal

In [ ]:
@dataclass
class Course:
    code: str
    title: str
    credits: int
    level: int
    department: str
    description: str
    seats_left: int
    delivery: Literal["in-person", "online", "hybrid"]

In [ ]:
COURSES = {
    "COMP 202": Course(
        code="COMP 202",
        title="Foundations of Programming",
        credits=3,
        level=200,
        department="Computer Science",
        description="Intro to programming in Python.",
        seats_left=12,
        delivery="in-person"
    ),

    "COMP 250": Course(
        code="COMP 250",
        title="Introduction to Computer Science",
        credits=3,
        level=200,
        department="Computer Science",
        description="Data structures and algorithms.",
        seats_left=5,
        delivery="in-person"
    ),

    "COMP 303": Course(
        code="COMP 303",
        title="Software Design",
        credits=3,
        level=300,
        department="Computer Science",
        description="Object-oriented design, testing, and software architecture.",
        seats_left=3,
        delivery="hybrid"
    ),

    "COMP 421": Course(
        code="COMP 421",
        title="Database Systems",
        credits=3,
        level=400,
        department="Computer Science",
        description="Relational databases, SQL, transactions, and query optimization.",
        seats_left=8,
        delivery="in-person"
    ),

    "COMP 551": Course(
        code="COMP 551",
        title="Applied Machine Learning",
        credits=4,
        level=500,
        department="Computer Science",
        description="Machine learning methods.",
        seats_left=0,
        delivery="in-person"
    ),

    "MATH 140": Course(
        code="MATH 140",
        title="Calculus 1",
        credits=3,
        level=100,
        department="Mathematics",
        description="Calculus fundamentals.",
        seats_left=20,
        delivery="in-person"
    ),

    "MATH 223": Course(
        code="MATH 223",
        title="Linear Algebra",
        credits=3,
        level=200,
        department="Mathematics",
        description="Matrices and vectors.",
        seats_left=15,
        delivery="in-person"
    ),

    "MATH 324": Course(
        code="MATH 324",
        title="Statistics",
        credits=3,
        level=300,
        department="Mathematics",
        description="Probability distributions, regression, and statistical inference.",
        seats_left=10,
        delivery="online"
    ),

    "MGCR 341": Course(
        code="MGCR 341",
        title="Introduction to Finance",
        credits=3,
        level=300,
        department="Management",
        description="Time value of money, capital budgeting, and portfolio theory.",
        seats_left=6,
        delivery="in-person"
    ),

    "ECSE 411": Course(
        code="ECSE 411",
        title="Operating Systems",
        credits=3,
        level=400,
        department="Electrical and Computer Engineering",
        description="Processes, threads, scheduling, memory management, and file systems.",
        seats_left=7,
        delivery="in-person"
    ),
}

In [ ]:
print(COURSES["COMP 551"])

Course(code='COMP 551', title='Applied Machine Learning', credits=4, level=500, department='Computer Science', description='Machine learning methods.', seats_left=0, delivery='in-person')


In [ ]:
PREREQUISITES = {
    "COMP 202": [],
    "COMP 250": ["COMP 202"],
    "COMP 303": ["COMP 250"],
    "COMP 421": ["COMP 250"],
    "COMP 551": ["COMP 250", "MATH 223", "MATH 324"],
    "MATH 140": [],
    "MATH 223": ["MATH 140"],
    "MATH 324": ["MATH 223"],
    "MGCR 341": [],
    "ECSE 411": ["COMP 250", "MATH 140"],
}

In [ ]:
@dataclass
class StudentProfile:
    name: str
    program: str
    completed_courses: list[str]
    max_credits_per_term: int = 15


DEFAULT_STUDENT = StudentProfile(
    name="Alex Chen",
    program="B.Sc. Computer Science",
    completed_courses=["COMP 202", "MATH 140", "MATH 223"],
    max_credits_per_term=15,
)

In [ ]:
POLICIES = {
    "max_credits_per_term": 15,
    "must_satisfy_prerequisites": True,
    "flag_full_courses": True,
    "human_advisor_required_for_substitution": True,
    "human_advisor_required_for_credit_transfer": True,
}

In [ ]:
print("Student:", DEFAULT_STUDENT.name)
print("Completed:", DEFAULT_STUDENT.completed_courses)
print("COMP 551 prerequisites:", PREREQUISITES["COMP 551"])
print("Max credits:", POLICIES["max_credits_per_term"])

Student: Alex Chen
Completed: ['COMP 202', 'MATH 140', 'MATH 223']
COMP 551 prerequisites: ['COMP 250', 'MATH 223']
Max credits: 15


In [ ]:
from agents import function_tool
import json
from typing import Optional

In [ ]:
@function_tool
def search_courses(
    keyword: Optional[str] = None,
    department: Optional[str] = None,
    level: Optional[int] = None,
    exclude_full: bool = False,
) -> str:
    """
    Search the fake McGill course catalog.

    Args:
        keyword: Search in course code, title, or description.
        department: Filter by department.
        level: Filter by course level, such as 200 or 500.
        exclude_full: If True, remove courses with 0 seats.

    Returns:
        A JSON string containing matching courses.
    """

    results = list(COURSES.values())

    if keyword:
        keyword_lower = keyword.lower()
        results = [
            course for course in results
            if keyword_lower in course.code.lower()
            or keyword_lower in course.title.lower()
            or keyword_lower in course.description.lower()
        ]

    if department:
        results = [
            course for course in results
            if course.department.lower() == department.lower()
        ]

    if level:
        results = [
            course for course in results
            if course.level == level
        ]

    if exclude_full:
        results = [
            course for course in results
            if course.seats_left > 0
        ]

    return json.dumps(
        {
            "found": len(results),
            "courses": [
                {
                    "code": course.code,
                    "title": course.title,
                    "credits": course.credits,
                    "level": course.level,
                    "department": course.department,
                    "description": course.description,
                    "seats_left": course.seats_left,
                    "delivery": course.delivery,
                    "prerequisites": PREREQUISITES.get(course.code, []),
                    "warning": "FULL - no seats available" if course.seats_left == 0 else None,
                }
                for course in results
            ],
        },
        indent=2,
    )

In [ ]:
def search_courses_test(
    keyword=None,
    department=None,
    level=None,
    exclude_full=False,
):
    import json

    results = list(COURSES.values())

    if keyword:
        keyword_lower = keyword.lower()
        results = [
            course for course in results
            if keyword_lower in course.code.lower()
            or keyword_lower in course.title.lower()
            or keyword_lower in course.description.lower()
        ]

    if department:
        results = [
            course for course in results
            if course.department.lower() == department.lower()
        ]

    if level:
        results = [
            course for course in results
            if course.level == level
        ]

    if exclude_full:
        results = [
            course for course in results
            if course.seats_left > 0
        ]

    return {
        "found": len(results),
        "courses": [
            {
                "code": course.code,
                "title": course.title,
                "seats_left": course.seats_left,
            }
            for course in results
        ],
    }

In [ ]:
search_courses_test(keyword="machine")

{'found': 1,
 'courses': [{'code': 'COMP 551',
   'title': 'Applied Machine Learning',
   'seats_left': 0}]}

In [ ]:
def check_prerequisites_test(course_code, completed_courses):
    course_code = course_code.upper().strip()

    if course_code not in COURSES:
        return {
            "eligible": False,
            "reason": f"{course_code} is not in the catalog."
        }

    required = PREREQUISITES.get(course_code, [])
    completed_set = set(completed_courses)

    missing = [
        course for course in required
        if course not in completed_set
    ]

    return {
        "course": course_code,
        "eligible": len(missing) == 0,
        "required_prerequisites": required,
        "missing_prerequisites": missing,
    }

In [ ]:
check_prerequisites_test(
    course_code="COMP 551",
    completed_courses=DEFAULT_STUDENT.completed_courses
)

{'course': 'COMP 551',
 'eligible': False,
 'required_prerequisites': ['COMP 250', 'MATH 223'],
 'missing_prerequisites': ['COMP 250']}

In [ ]:
check_prerequisites_test(
    course_code="MATH 223",
    completed_courses=DEFAULT_STUDENT.completed_courses
)

{'course': 'MATH 223',
 'eligible': True,
 'required_prerequisites': ['MATH 140'],
 'missing_prerequisites': []}

In [ ]:
check_prerequisites_test(
    course_code="COMP 999",
    completed_courses=DEFAULT_STUDENT.completed_courses
)

{'eligible': False, 'reason': 'COMP 999 is not in the catalog.'}

In [ ]:
def flag_policy_risk_test(
    proposed_courses,
    completed_courses,
    current_term_credits=0
):
    violations = []
    requires_human_escalation = False

    # 1. Calculate total credits
    proposed_credits = 0

    for code in proposed_courses:
        code = code.upper().strip()

        if code in COURSES:
            proposed_credits += COURSES[code].credits
        else:
            violations.append({
                "policy": "POLICY-UNKNOWN-COURSE",
                "course": code,
                "detail": f"{code} is not in the catalog.",
                "requires_escalation": False
            })

    total_credits = proposed_credits + current_term_credits

    # 2. Check prerequisites
    for code in proposed_courses:
        code = code.upper().strip()

        if code not in COURSES:
            continue

        prereq_result = check_prerequisites_test(
            course_code=code,
            completed_courses=completed_courses
        )

        if not prereq_result["eligible"]:
            violations.append({
                "policy": "POLICY-PREREQ",
                "course": code,
                "detail": f"Missing prerequisites: {prereq_result['missing_prerequisites']}",
                "requires_escalation": False
            })

    # 3. Check full courses
    for code in proposed_courses:
        code = code.upper().strip()

        if code in COURSES and COURSES[code].seats_left == 0:
            violations.append({
                "policy": "POLICY-SEATS",
                "course": code,
                "detail": f"{code} is full with 0 seats left.",
                "requires_escalation": False
            })

    # 4. Check credit overload
    if total_credits >= POLICIES["max_credits_per_term"]:
        requires_human_escalation = True
        violations.append({
            "policy": "POLICY-OVERLOAD",
            "course": None,
            "detail": f"Total credits = {total_credits}, which is at or above the 15-credit cap.",
            "requires_escalation": True
        })

    return {
        "proposed_courses": proposed_courses,
        "proposed_credits": proposed_credits,
        "total_term_credits": total_credits,
        "violations": violations,
        "requires_human_escalation": requires_human_escalation,
    }

In [ ]:
flag_policy_risk_test(
    proposed_courses=["COMP 250", "MATH 324"],
    completed_courses=DEFAULT_STUDENT.completed_courses
)

{'proposed_courses': ['COMP 250', 'MATH 324'],
 'proposed_credits': 3,
 'total_term_credits': 3,
 'violations': [{'policy': 'POLICY-UNKNOWN-COURSE',
   'course': 'MATH 324',
   'detail': 'MATH 324 is not in the catalog.',
   'requires_escalation': False}],
 'requires_human_escalation': False}

In [ ]:
flag_policy_risk_test(
    proposed_courses=["COMP 551"],
    completed_courses=DEFAULT_STUDENT.completed_courses
)

{'proposed_courses': ['COMP 551'],
 'proposed_credits': 4,
 'total_term_credits': 4,
 'violations': [{'policy': 'POLICY-PREREQ',
   'course': 'COMP 551',
   'detail': "Missing prerequisites: ['COMP 250']",
   'requires_escalation': False},
  {'policy': 'POLICY-SEATS',
   'course': 'COMP 551',
   'detail': 'COMP 551 is full with 0 seats left.',
   'requires_escalation': False}],
 'requires_human_escalation': False}

In [ ]:
flag_policy_risk_test(
    proposed_courses=["COMP 250", "MATH 324", "MGCR 341", "COMP 202", "MATH 140"],
    completed_courses=DEFAULT_STUDENT.completed_courses
)

{'proposed_courses': ['COMP 250',
  'MATH 324',
  'MGCR 341',
  'COMP 202',
  'MATH 140'],
 'proposed_credits': 9,
 'total_term_credits': 9,
 'violations': [{'policy': 'POLICY-UNKNOWN-COURSE',
   'course': 'MATH 324',
   'detail': 'MATH 324 is not in the catalog.',
   'requires_escalation': False},
  {'policy': 'POLICY-UNKNOWN-COURSE',
   'course': 'MGCR 341',
   'detail': 'MGCR 341 is not in the catalog.',
   'requires_escalation': False}],
 'requires_human_escalation': False}

In [ ]:
flag_policy_risk_test(
    proposed_courses=["COMP 999"],
    completed_courses=DEFAULT_STUDENT.completed_courses
)

{'proposed_courses': ['COMP 999'],
 'proposed_credits': 0,
 'total_term_credits': 0,
 'violations': [{'policy': 'POLICY-UNKNOWN-COURSE',
   'course': 'COMP 999',
   'detail': 'COMP 999 is not in the catalog.',
   'requires_escalation': False}],
 'requires_human_escalation': False}

In [ ]:
flag_policy_risk_test(
    proposed_courses=["COMP 999"],
    completed_courses=DEFAULT_STUDENT.completed_courses
)

{'proposed_courses': ['COMP 999'],
 'proposed_credits': 0,
 'total_term_credits': 0,
 'violations': [{'policy': 'POLICY-UNKNOWN-COURSE',
   'course': 'COMP 999',
   'detail': 'COMP 999 is not in the catalog.',
   'requires_escalation': False}],
 'requires_human_escalation': False}

In [ ]:
@function_tool
def search_courses(
    keyword: Optional[str] = None,
    department: Optional[str] = None,
    level: Optional[int] = None,
    exclude_full: bool = False,
) -> str:
    """
    Search the fake McGill course catalog.

    Args:
        keyword: Search in course code, title, or description.
        department: Filter by department name.
        level: Filter by course level, such as 200 or 500.
        exclude_full: If True, remove courses with 0 seats.

    Returns:
        JSON string with matching courses.
    """

    result = search_courses_test(
        keyword=keyword,
        department=department,
        level=level,
        exclude_full=exclude_full,
    )

    return json.dumps(result, indent=2)

In [ ]:
@function_tool
def check_prerequisites(
    course_code: str,
    completed_courses: list[str],
) -> str:
    """
    Check whether a student satisfies prerequisites for a course.

    Args:
        course_code: Course code, such as COMP 551.
        completed_courses: Courses the student has already completed.

    Returns:
        JSON string with eligibility and missing prerequisites.
    """

    result = check_prerequisites_test(
        course_code=course_code,
        completed_courses=completed_courses,
    )

    return json.dumps(result, indent=2)

In [ ]:
@function_tool
def flag_policy_risk(
    proposed_courses: list[str],
    completed_courses: list[str],
    current_term_credits: int = 0,
) -> str:
    """
    Check a proposed course schedule against advising policy rules.

    Args:
        proposed_courses: Courses the student wants to take.
        completed_courses: Courses the student has already completed.
        current_term_credits: Credits already registered this term.

    Returns:
        JSON string with violations and escalation status.
    """

    result = flag_policy_risk_test(
        proposed_courses=proposed_courses,
        completed_courses=completed_courses,
        current_term_credits=current_term_credits,
    )

    return json.dumps(result, indent=2)

In [ ]:
print(search_courses)
print(check_prerequisites)
print(flag_policy_risk)

FunctionTool(name='search_courses', description='Search the fake McGill course catalog.', params_json_schema={'properties': {'keyword': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'description': 'Search in course code, title, or description.', 'title': 'Keyword'}, 'department': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'description': 'Filter by department name.', 'title': 'Department'}, 'level': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'description': 'Filter by course level, such as 200 or 500.', 'title': 'Level'}, 'exclude_full': {'default': False, 'description': 'If True, remove courses with 0 seats.', 'title': 'Exclude Full', 'type': 'boolean'}}, 'title': 'search_courses_args', 'type': 'object', 'additionalProperties': False, 'required': ['keyword', 'department', 'level', 'exclude_full']}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x79c6a44dbce0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

In [ ]:
class CourseRecommendation(BaseModel):
    recommended_courses: list[str] = Field(
        description="Course codes recommended for the upcoming term."
    )

    rationale: str = Field(
        description="Short explanation of why these courses are recommended."
    )

    risks: list[str] = Field(
        description="Risks such as missing prerequisites, full courses, or overload."
    )

    requires_advisor_approval: bool = Field(
        description="Whether a human advisor must approve this recommendation."
    )

    advisor_approval_reason: Optional[str] = Field(
        default=None,
        description="Reason why advisor approval is required, if any."
    )

    credits_this_term: int = Field(
        description="Total number of credits in the recommended schedule."
    )

In [ ]:
sample_output = CourseRecommendation(
    recommended_courses=["COMP 250", "MATH 324"],
    rationale="COMP 250 is the next CS course after COMP 202. MATH 324 supports future machine learning coursework.",
    risks=[],
    requires_advisor_approval=False,
    advisor_approval_reason=None,
    credits_this_term=6,
)

sample_output

CourseRecommendation(recommended_courses=['COMP 250', 'MATH 324'], rationale='COMP 250 is the next CS course after COMP 202. MATH 324 supports future machine learning coursework.', risks=[], requires_advisor_approval=False, advisor_approval_reason=None, credits_this_term=6)

In [ ]:
print(sample_output.model_dump_json(indent=2))

{
  "recommended_courses": [
    "COMP 250",
    "MATH 324"
  ],
  "rationale": "COMP 250 is the next CS course after COMP 202. MATH 324 supports future machine learning coursework.",
  "risks": [],
  "requires_advisor_approval": false,
  "advisor_approval_reason": null,
  "credits_this_term": 6
}


In [ ]:
class EscalationCheck(BaseModel):
    requires_escalation: bool
    reason: str

In [ ]:
from agents import Agent

In [ ]:
escalation_detector = Agent(
    name="Escalation Detector",

    instructions="""
You determine whether a student's request must be handled
by a human advisor.

Escalate if the request involves:

- course substitutions
- course equivalencies
- degree requirement exceptions
- transfer credits
- grade appeals

For normal course selection questions,
return requires_escalation=False.
""",

    model="gpt-4o-mini",

    output_type=EscalationCheck,
)

In [ ]:
test_guardrail = EscalationCheck(
    requires_escalation=True,
    reason="Course substitution request."
)

test_guardrail

EscalationCheck(requires_escalation=True, reason='Course substitution request.')

Named Guardrail:
GUARDRAIL_ESCALATION_REQUEST

Purpose:
Prevent the agent from making autonomous decisions
about course substitutions, transfer credits,
degree exceptions, and grade appeals.

Action:
Block the recommendation workflow and direct the
student to a human academic advisor.


In [ ]:
example_guardrail_cases = [
    "Can MATH 324 substitute for a required probability course?",
    "Can I transfer credits from another university?",
    "Can I appeal my grade?",
]

example_normal_cases = [
    "What courses should I take next term?",
    "Can I take COMP 250?",
    "What are the prerequisites for COMP 551?",
]

print("Guardrail examples:")
print(example_guardrail_cases)

print("\nNormal advising examples:")
print(example_normal_cases)

Guardrail examples:
['Can MATH 324 substitute for a required probability course?', 'Can I transfer credits from another university?', 'Can I appeal my grade?']

Normal advising examples:
['What courses should I take next term?', 'Can I take COMP 250?', 'What are the prerequisites for COMP 551?']


In [ ]:
from agents import Agent

In [ ]:
example_guardrail_cases = [
    "Can MATH 324 substitute for a required probability course?",
    "Can I transfer credits from another university?",
    "Can I appeal my grade?",
]

example_normal_cases = [
    "What courses should I take next term?",
    "Can I take COMP 250?",
    "What are the prerequisites for COMP 551?",
]

print("Guardrail examples:")
print(example_guardrail_cases)

print("\nNormal advising examples:")
print(example_normal_cases)

Guardrail examples:
['Can MATH 324 substitute for a required probability course?', 'Can I transfer credits from another university?', 'Can I appeal my grade?']

Normal advising examples:
['What courses should I take next term?', 'Can I take COMP 250?', 'What are the prerequisites for COMP 551?']


In [ ]:
advisor_agent = Agent(
    name="McGill Course Advisor",

    instructions="""
You are a McGill University course advisor.

IMPORTANT TOOL RULES:
1. You must call search_courses exactly once at the beginning.
2. You must call check_prerequisites only for the courses you plan to recommend.
3. You must call flag_policy_risk exactly once before final output.
4. After calling flag_policy_risk, immediately stop and return the final CourseRecommendation.
5. Never call search_courses more than once.
6. Recommend at most 2 courses.

Fake catalog:
- COMP 202: Foundations of Programming, 3 credits, no prerequisites
- COMP 250: Introduction to Computer Science, 3 credits, requires COMP 202
- COMP 303: Software Design, 3 credits, requires COMP 250
- COMP 421: Database Systems, 3 credits, requires COMP 250
- COMP 551: Applied Machine Learning, 4 credits, requires COMP 250, MATH 223, and MATH 324, currently full
- MATH 140: Calculus 1, 3 credits, no prerequisites
- MATH 223: Linear Algebra, 3 credits, requires MATH 140
- MATH 324: Statistics, 3 credits, requires MATH 223
- MGCR 341: Introduction to Finance, 3 credits, no prerequisites
- ECSE 411: Operating Systems, 3 credits, requires COMP 250 and MATH 140

Recommendation rules:
- If the student completed COMP 202, COMP 250 is the natural next CS course.
- If the student wants machine learning, recommend COMP 250 and/or MATH 324 before COMP 551.
- Do not recommend COMP 551 directly if prerequisites are missing.
- If COMP 551 is discussed, mention that it is full.

Escalate and stop if the request involves:
- course substitutions
- course equivalencies
- transfer credits
- degree exceptions
- grade appeals

Always return a CourseRecommendation object.
""",

    model="gpt-4o-mini",

    tools=[
        search_courses,
        check_prerequisites,
        flag_policy_risk,
    ],

    output_type=CourseRecommendation,

    input_guardrails=[
        escalation_guardrail,
    ],
)

In [ ]:
advisor_agent

Agent(name='McGill Course Advisor', handoff_description=None, tools=[FunctionTool(name='search_courses', description='Search the fake McGill course catalog.', params_json_schema={'properties': {'keyword': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'description': 'Search in course code, title, or description.', 'title': 'Keyword'}, 'department': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'description': 'Filter by department name.', 'title': 'Department'}, 'level': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'description': 'Filter by course level, such as 200 or 500.', 'title': 'Level'}, 'exclude_full': {'default': False, 'description': 'If True, remove courses with 0 seats.', 'title': 'Exclude Full', 'type': 'boolean'}}, 'title': 'search_courses_args', 'type': 'object', 'additionalProperties': False, 'required': ['keyword', 'department', 'level', 'exclude_full']}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x79c6a44dbce0>, strict_json_

In [ ]:
from agents import Runner

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "xxxx"

In [ ]:
result = await Runner.run(
    advisor_agent,
    "I completed COMP 202. Can I take COMP 250?",
    max_turns=20,
)

result.final_output

CourseRecommendation(recommended_courses=['COMP 250'], rationale='Since you have completed COMP 202, you are eligible to take COMP 250 as the next course.', risks=[], requires_advisor_approval=False, advisor_approval_reason=None, credits_this_term=3)

In [ ]:
from agents import Runner

guardrail_test = await Runner.run(
    escalation_detector,
    "Can MATH 324 substitute for my required probability course?"
)

guardrail_test.final_output

EscalationCheck(requires_escalation=True, reason='Course substitution request.')

In [ ]:
normal_test = await Runner.run(
    escalation_detector,
    "Can I take COMP 250 after completing COMP 202?"
)

normal_test.final_output

EscalationCheck(requires_escalation=False, reason='Normal course selection question.')

In [ ]:
from agents import (
    input_guardrail,
    GuardrailFunctionOutput,
    RunContextWrapper,
    InputGuardrailTripwireTriggered,
)

In [ ]:
@input_guardrail
async def escalation_guardrail(ctx, agent, input):
    """
    GUARDRAIL_ESCALATION_REQUEST

    Blocks requests involving:
    - course substitutions
    - course equivalencies
    - transfer credits
    - degree exceptions
    - grade appeals
    """

    result = await Runner.run(
        escalation_detector,
        input
    )

    check = result.final_output

    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=check.requires_escalation,
    )

In [ ]:
try:
    blocked_result = await Runner.run(
        advisor_agent,
        "Can MATH 324 substitute for my required probability course?",
        max_turns=20,
    )

    print(blocked_result.final_output)

except InputGuardrailTripwireTriggered as e:
    print("GUARDRAIL BLOCKED THE REQUEST")
    print(e)

GUARDRAIL BLOCKED THE REQUEST
Guardrail InputGuardrail triggered tripwire


In [ ]:
async def run_case(case_id, query):
    print("=" * 60)
    print(case_id)
    print("Query:", query)

    try:
        result = await Runner.run(
            advisor_agent,
            query,
            max_turns=30,
        )

        output = result.final_output

        print("PASSED")
        print(output.model_dump_json(indent=2))

        return {
            "case": case_id,
            "query": query,
            "status": "PASSED",
            "output": output.model_dump(),
        }

    except InputGuardrailTripwireTriggered:
        print("GUARDRAIL BLOCKED")

        return {
            "case": case_id,
            "query": query,
            "status": "GUARDRAIL_BLOCKED",
        }

    except Exception as e:
        print("FAILED")
        print(str(e))

        return {
            "case": case_id,
            "query": query,
            "status": "FAILED",
            "error": str(e),
        }

In [ ]:
eval_outputs = {}

eval_outputs["EVAL-01"] = await run_case(
    "EVAL-01 Happy path",
    "I completed COMP 202, MATH 140, and MATH 223. What CS courses should I take next term?"
)

eval_outputs["EVAL-02"] = await run_case(
    "EVAL-02 ML path with missing prerequisite",
    "I completed COMP 202, MATH 140, and MATH 223. I want to take COMP 551. What do I need first?"
)

eval_outputs["EVAL-03"] = await run_case(
    "EVAL-03 Full course risk",
    "I completed COMP 202, COMP 250, MATH 140, MATH 223, and MATH 324. Can I take COMP 551 this term?"
)

eval_outputs["EVAL-04"] = await run_case(
    "EVAL-04 Overload risk",
    "I want to take COMP 250, MATH 324, MGCR 341, COMP 202, and MATH 140 this term. Is that okay?"
)

eval_outputs["EVAL-05"] = await run_case(
    "EVAL-05 Missing prerequisite",
    "I only completed COMP 202. Can I take COMP 303 next term?"
)

eval_outputs["EVAL-06"] = await run_case(
    "EVAL-06 Guardrail substitution request",
    "Can MATH 324 substitute for my required probability course?"
)

eval_outputs["EVAL-07"] = await run_case(
    "EVAL-07 Unknown course",
    "What are the prerequisites for COMP 999?"
)

EVAL-01 Happy path
Query: I completed COMP 202, MATH 140, and MATH 223. What CS courses should I take next term?
PASSED
{
  "recommended_courses": [
    "COMP 250",
    "MATH 324"
  ],
  "rationale": "These courses build on your completed courses and are prerequisites for advanced studies, especially if you're interested in machine learning.",
  "risks": [],
  "requires_advisor_approval": false,
  "advisor_approval_reason": null,
  "credits_this_term": 6
}
EVAL-02 ML path with missing prerequisite
Query: I completed COMP 202, MATH 140, and MATH 223. I want to take COMP 551. What do I need first?
PASSED
{
  "recommended_courses": [
    "COMP 250",
    "MATH 324"
  ],
  "rationale": "To take COMP 551, you need to complete COMP 250 and MATH 324 first. After taking these courses, you will meet the prerequisites for COMP 551.",
  "risks": [],
  "requires_advisor_approval": false,
  "advisor_approval_reason": null,
  "credits_this_term": 6
}
EVAL-03 Full course risk
Query: I completed COMP 2

In [ ]:
import json

with open("eval_results.json", "w") as f:
    json.dump(eval_outputs, f, indent=2)

print("Saved eval_results.json")

Saved eval_results.json


In [ ]:
eval_outputs

{'EVAL-01': {'case': 'EVAL-01 Happy path',
  'query': 'I completed COMP 202, MATH 140, and MATH 223. What CS courses should I take next term?',
  'status': 'PASSED',
  'output': {'recommended_courses': ['COMP 250', 'MATH 324'],
   'rationale': "These courses build on your completed courses and are prerequisites for advanced studies, especially if you're interested in machine learning.",
   'risks': [],
   'requires_advisor_approval': False,
   'advisor_approval_reason': None,
   'credits_this_term': 6}},
 'EVAL-02': {'case': 'EVAL-02 ML path with missing prerequisite',
  'query': 'I completed COMP 202, MATH 140, and MATH 223. I want to take COMP 551. What do I need first?',
  'status': 'PASSED',
  'output': {'recommended_courses': ['COMP 250', 'MATH 324'],
   'rationale': 'To take COMP 551, you need to complete COMP 250 and MATH 324 first. After taking these courses, you will meet the prerequisites for COMP 551.',
   'risks': [],
   'requires_advisor_approval': False,
   'advisor_appro

In [ ]:
from google.colab import files
files.download("eval_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
readme = """
# McGill Course Advisor Agent

## Overview

This project implements a single-agent academic advising system using the OpenAI Agents SDK.

The agent helps students:

- Search courses
- Check prerequisites
- Identify policy risks
- Determine when advisor approval is required

The system operates on a small synthetic McGill course catalog.

---

## Agent Design

Agent:
- McGill Course Advisor

Typed Tools:
1. search_courses
2. check_prerequisites
3. flag_policy_risk

Structured Output:
- CourseRecommendation

Guardrail:
- GUARDRAIL_ESCALATION_REQUEST

The guardrail blocks:
- course substitutions
- transfer credits
- degree exceptions
- grade appeals

---

## State Strategy

Stored:
- Course catalog
- Prerequisite table
- Policy rules
- Student profile

Recomputed:
- Eligibility checks
- Risk analysis
- Recommendations

---

## Eval Cases

1. Happy path
2. ML path with missing prerequisite
3. Full course risk
4. Overload risk
5. Missing prerequisite
6. Guardrail substitution request
7. Unknown course

---

## Running

Install:

pip install openai-agents pydantic

Set API key:

export OPENAI_API_KEY=YOUR_KEY

Run notebook cells.

---

## Evidence

- Structured outputs
- Eval traces
- Guardrail examples
"""

In [ ]:
with open("README.md", "w") as f:
    f.write(readme)

print("README.md created")

README.md created


In [ ]:
reflection = """
# Reflection

## What Worked

The agent successfully used typed tools to check prerequisites and identify policy risks.

Structured outputs made evaluation easier because each response followed the same schema.

The guardrail correctly blocked course substitution requests before the advisor agent was executed.

## What Failed

Initially, the search_courses tool caused tool loops and produced MaxTurnsExceeded errors.

The first version of the course catalog was incomplete, causing the agent to incorrectly report that some courses did not exist.

One evaluation case produced an inconsistency where overload risk was detected but credits_this_term was returned as 0.

## Improvements

The tool loop issue was reduced by limiting tool usage and simplifying agent instructions.

The course catalog and prerequisite table were expanded and corrected.

Additional evaluation cases were added to test edge conditions.

## Remaining Risks

The catalog is synthetic and much smaller than the real McGill catalog.

There is no timetable conflict detection.

The system does not track graduation requirements.

The overload calculation should be validated independently from the language model output.

## Future Work

Connect to a real course catalog API.

Add timetable conflict checking.

Add degree progress tracking.

Add stronger validation for structured outputs.
"""

In [ ]:
with open("REFLECTION.md", "w") as f:
    f.write(reflection)

print("REFLECTION.md created")

REFLECTION.md created


In [ ]:
from google.colab import files

files.download("README.md")
files.download("REFLECTION.md")
files.download("eval_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>